# Piper TTS Voice Training for Speech2Text App

This notebook trains a **Piper TTS voice model** using recordings from your Speech2Text Android app.

## What is Piper?
Piper is a fast, local neural text-to-speech system that:
- ✅ Runs completely offline
- ✅ Works on Android devices
- ✅ Produces high-quality speech
- ✅ Supports multiple languages (German, English, Polish, etc.)

## Prerequisites:
- Exported training data from Speech2Text app to Google Drive
- At least 30-60 minutes of clear audio recordings
- GPU runtime enabled in Colab (Runtime → Change runtime type → GPU)

## Training Time:
- With checkpoint (recommended): 2-4 hours
- From scratch: 8-12+ hours

## Steps:
1. Setup GPU and system packages
2. Clone and install Piper training tools
3. Mount Google Drive and prepare data
4. Configure training parameters
5. Start training
6. Export trained model
7. Download for use in your app

## Step 1: Check GPU and System Info

In [2]:
# Check GPU availability
import torch, platform, sys

print("📊 System Information:")
print(f"   Python: {sys.version.split()[0]}")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"\n✅ GPU Details:")
    !nvidia-smi
else:
    print("\n⚠️ No GPU detected!")
    print("   Please enable GPU: Runtime → Change runtime type → GPU → Save")
    print("   Then restart this notebook.")

📊 System Information:
   Python: 3.12.12
   PyTorch: 2.10.0+cpu
   CUDA available: False

⚠️ No GPU detected!
   Please enable GPU: Runtime → Change runtime type → GPU → Save
   Then restart this notebook.


## Step 2: Install System Packages

In [3]:
# Install required system packages including eSpeak
print("📦 Installing system packages...\n")

!sudo apt-get update -y
!sudo apt-get install -y build-essential cmake ninja-build espeak-ng espeak-ng-data libespeak-ng-dev pkg-config ffmpeg

# Verify eSpeak installation
print("\n✅ Checking eSpeak-NG version:")
!pkg-config --modversion espeak-ng

📦 Installing system packages...

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,815 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,780 kB]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-sec

## Step 3: Clone Piper Training Repository

In [4]:
# Clone the Piper GPL training repository
import os

os.chdir('/content')

# Remove if exists
if os.path.exists('piper1-gpl'):
    !rm -rf piper1-gpl

print("📥 Cloning Piper training repository...\n")
!git clone https://github.com/OHF-voice/piper1-gpl.git

os.chdir('piper1-gpl')
print(f"\n✅ Repository cloned to: {os.getcwd()}")

📥 Cloning Piper training repository...

Cloning into 'piper1-gpl'...
remote: Enumerating objects: 1130, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 1130 (delta 26), reused 15 (delta 15), pack-reused 1073 (from 2)
Receiving objects: 100% (1130/1130), 4.74 MiB | 19.49 MiB/s, done.
Resolving deltas: 100% (637/637), done.

✅ Repository cloned to: /content/piper1-gpl


## Step 4: Install Python Dependencies

In [6]:
# Install Piper in editable mode with training dependencies
print("📦 Installing Python dependencies...\n")
print("   This may take 3-5 minutes...\n")

!python3 -m pip install --upgrade pip setuptools wheel
!python3 -m pip install -e ".[train]"

print("\n✅ Python dependencies installed!")

📦 Installing Python dependencies...

   This may take 3-5 minutes...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.6 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
Obtaining file:///content/piper1-gpl
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproje

## Step 5: Build Alignment Extension

In [7]:
# Build the Cython extension for monotonic alignment
os.chdir('/content/piper1-gpl')

print("🔨 Building alignment extension...\n")

!chmod +x ./build_monotonic_align.sh
!./build_monotonic_align.sh

print("\n✅ Alignment extension built!")

🔨 Building alignment extension...

Compiling /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx because it changed.
[1/1] Cythonizing /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx
/usr/local/lib/python3.12/dist-packages/Cython/Compiler/Main.py:381: FutureWarning: Cython directive 'language_level' not set, using '3str' for now (Py3). This has changed from earlier releases! File: /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx
  tree = Parsing.p_module(s, pxd, full_module_name)
performance hint: core.pyx:7:5: Exception check on 'maximum_path_each' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'maximum_path_each' as 'noexcept' if you control the definition and you're sure you don't want the function to raise exceptions.
	2. Use an 'int' return type on 'maximum_path_each' to allow an error code to be returned.
performance hint: core.pyx:38:6: Exception check on 'maximum_path_c' will always require the GIL to

## Step 6: Development Build

In [8]:
# Install additional build tools
!python3 -m pip install --upgrade pip setuptools wheel scikit-build cmake ninja

  Using cached scikit_build-0.19.0-py3-none-any.whl.metadata (19 kB)
  Using cached cmake-4.2.3-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (6.5 kB)
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
Using cached scikit_build-0.19.0-py3-none-any.whl (85 kB)
Using cached cmake-4.2.3-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (28.9 MB)
Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (180 kB)
  Attempting uninstall: cmake
    Found existing installation: cmake 3.31.10
    Uninstalling cmake-3.31.10:
      Successfully uninstalled cmake-3.31.10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-build]


In [9]:
# Build extensions in place
os.chdir('/content/piper1-gpl')

print("🔨 Building Piper extensions...\n")
!python3 setup.py build_ext --inplace -v

print("\n✅ Build complete!")

🔨 Building Piper extensions...



--------------------------------------------------------------------------------
-- Trying 'Ninja' generator
--------------------------------
---------------------------
----------------------
-----------------
------------
-------
--
Not searching for unused variables given on the command line.
-- The C compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- The CXX compiler identification is GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Configuring done (0.5s)
-- Generating done (0.0s)
-- Build files have been written to: /content/piper1-gpl/_cmake_test_compile/build
--
-------
------------
----

## Step 7: Mount Google Drive

In [10]:
from google.colab import drive
import os

print("📂 Mounting Google Drive...\n")
drive.mount('/content/drive')

# Verify training data location
TRAINING_DATA_PATH = '/content/drive/MyDrive/TTS_Voice_Samples'

if os.path.exists(TRAINING_DATA_PATH):
    wav_files = [f for f in os.listdir(TRAINING_DATA_PATH) if f.endswith('.wav')]
    csv_files = [f for f in os.listdir(TRAINING_DATA_PATH) if f.endswith('.csv')]

    print(f"\n✅ Found training data:")
    print(f"   WAV files: {len(wav_files)}")
    print(f"   CSV files: {len(csv_files)}")
else:
    print(f"\n❌ Training data not found at: {TRAINING_DATA_PATH}")
    print(f"   Please export data from your Speech2Text app first!")

📂 Mounting Google Drive...

Mounted at /content/drive

✅ Found training data:
   WAV files: 30
   CSV files: 1


## Step 8: Prepare Training Data

In [21]:
import re
from pathlib import Path
from collections import Counter

# 1. Setup
# Reverting to the pattern that correctly captures the full language name from the filename
# Example: 'voice_🇵🇱 Polski_1772658337076.wav' -> 'Polski'
pattern = re.compile(r"_.+?\s([a-zA-Z]+)_")
language_codes = []

# 2. Process
path_obj = Path(TRAINING_DATA_PATH)
if not path_obj.exists():
    raise FileNotFoundError(f"❌ Directory not found: {TRAINING_DATA_PATH}")

for file_path in path_obj.glob("*.wav"):
    match = pattern.search(file_path.name)
    if match:
        language_codes.append(match.group(1))

# 3. The "Stop" Guard
if len(language_codes) == 0:
    raise Exception(f"❌ Found 0 matches in {TRAINING_DATA_PATH}. Stopping script to prevent errors downstream.")

# 4. Success
stats = Counter(language_codes)
print(f"✅ Success! Found {len(language_codes)} files.")
ESPEAK_VOICE = language_codes[0]
print(f"   Detected language: {ESPEAK_VOICE}")

✅ Success! Found 30 files.
   Detected language: Polski


In [22]:
from pathlib import Path
import pandas as pd
import shutil
import os # Added import for os module

# Setup paths
DATA_ROOT = Path("/content/drive/MyDrive/piper_training")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

AUDIO_DIR = DATA_ROOT / "wavs"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

METADATA_CSV = DATA_ROOT / "metadata.csv"

print("📦 Preparing training data...\n")

# Read CSV from app export
csv_files = [f for f in os.listdir(TRAINING_DATA_PATH) if f.endswith('.csv')]

if csv_files:
    app_csv = os.path.join(TRAINING_DATA_PATH, csv_files[0])

    # Read app CSV: filename|text|language
    # Using 'utf-8' encoding, as files on disk and original CSV are likely UTF-8.
    # The previous 'latin-1' caused mismatch for filenames with emojis.
    with open(app_csv, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    # Convert to Piper format: filename|text (no language column)
    piper_lines = []
    for line in lines:
        parts = line.strip().split('|')
        if len(parts) >= 2:
            filename = parts[0]
            text = parts[1]

            # Copy WAV file
            src = Path(TRAINING_DATA_PATH) / filename
            dst = AUDIO_DIR / filename

            if src.exists():
                shutil.copy2(src, dst)
                piper_lines.append(f"{filename}|{text}")

    # Write Piper metadata.csv
    with open(METADATA_CSV, 'w', encoding='utf-8') as f: # Always write to UTF-8 for consistency
        f.write('\n'.join(piper_lines))

    print(f"✅ Prepared {len(piper_lines)} training samples")
    print(f"   Audio files: {AUDIO_DIR}")
    print(f"   Metadata: {METADATA_CSV}")

    # Show sample
    if piper_lines:
        print(f"\n📝 Sample entry:")
        print(f"   {piper_lines[0][:80]}...")
else:
    print("❌ No CSV file found in training data!")

📦 Preparing training data...

✅ Prepared 30 training samples
   Audio files: /content/drive/MyDrive/piper_training/wavs
   Metadata: /content/drive/MyDrive/piper_training/metadata.csv

📝 Sample entry:
   voice_🇵🇱 Polski_1772658337076.wav|Słońce już wysoko, a poranne mgły powoli znika...


## Step 9: Configure Training Parameters

In [23]:
from pathlib import Path

# ==== AUTO-DETECT LANGUAGE FROM RECORDINGS (Using value from previous step) ====
# The `ESPEAK_VOICE` variable is already set by the previous cell (MYm-hGqDZi7j)
# and contains the 'long' language name (e.g., 'Polski', 'English', 'Deutsch').

# Map the 'long' language name to its eSpeak short code for checkpoint lookup
long_to_espeak_code_map = {
    'Polski': 'pl',
    'English': 'en-us',
    'Deutsch': 'de'
}

# Checkpoint URLs for different languages
checkpoint_map = {
    'de': 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/de/de_DE/thorsten/medium/epoch%3D2164-step%3D1355540.ckpt',
    'en-us': 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt',
    'pl': ''  # No medium checkpoint available for Polish
}

# ==== TRAINING CONFIGURATION ====

# Voice settings
VOICE_NAME = "my_own_voice"

# Determine the eSpeak short code from the previously detected long name
espeak_short_code = long_to_espeak_code_map.get(ESPEAK_VOICE, 'de') # Default to 'de' if not found

# Set the CKPT_URL based on the determined short code
CKPT_URL = checkpoint_map.get(espeak_short_code, '')

# Update ESPEAK_VOICE to the short code for consistency in later steps (e.g., the training command)
ESPEAK_VOICE = espeak_short_code

# Audio settings
SAMPLE_RATE_HZ = 22050

# Training settings
BATCH_SIZE = 8  # Reduce to 4 if you get out of memory errors

# Paths
CACHE_DIR = Path("/content/piper_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = DATA_ROOT / f"{VOICE_NAME}.json"

print("⚙️ Training Configuration:")
print(f"   Voice name: {VOICE_NAME}")
print(f"   Language (eSpeak): {ESPEAK_VOICE}")
print(f"   Sample rate: {SAMPLE_RATE_HZ} Hz")
print(f"   Batch size: {BATCH_SIZE}")
if CKPT_URL:
    print(f"   Using checkpoint: Yes (faster training)")
else:
    print(f"   Using checkpoint: No (training from scratch, slower)")
    print(f"   ⚠️ Note: Polish has no pre-trained checkpoint, training will take longer")
print(f"\n📁 Paths:")
print(f"   CSV: {METADATA_CSV.exists()} - {METADATA_CSV}")
print(f"   Audio dir: {AUDIO_DIR.exists()} - {AUDIO_DIR}")
print(f"   Cache: {CACHE_DIR}")
print(f"   Config: {CONFIG_PATH}")

⚙️ Training Configuration:
   Voice name: my_own_voice
   Language (eSpeak): pl
   Sample rate: 22050 Hz
   Batch size: 8
   Using checkpoint: No (training from scratch, slower)
   ⚠️ Note: Polish has no pre-trained checkpoint, training will take longer

📁 Paths:
   CSV: True - /content/drive/MyDrive/piper_training/metadata.csv
   Audio dir: True - /content/drive/MyDrive/piper_training/wavs
   Cache: /content/piper_cache
   Config: /content/drive/MyDrive/piper_training/my_own_voice.json


## Step 10: Verify eSpeak Voice

In [24]:
# Check available eSpeak voices
print("🗣️ Available eSpeak voices:\n")
!espeak-ng --voices | grep -E "de|en|pl" | head -n 20

print(f"\n✅ Selected voice: {ESPEAK_VOICE}")
print(f"   Make sure this matches your recording language!")

🗣️ Available eSpeak voices:

Pty Language       Age/Gender VoiceName          File                 Other Languages
 5  bn              --/M      Bengali            inc/bn               
 5  de              --/M      German             gmw/de               
 5  en-029          --/M      English_(Caribbean) gmw/en-029           (en 10)
 2  en-gb           --/M      English_(Great_Britain) gmw/en               (en 2)
 5  en-gb-scotland  --/M      English_(Scotland) gmw/en-GB-scotland   (en 4)
 5  en-gb-x-gbclan  --/M      English_(Lancaster) gmw/en-GB-x-gbclan   (en-gb 3)(en 5)
 5  en-gb-x-gbcwmd  --/M      English_(West_Midlands) gmw/en-GB-x-gbcwmd   (en-gb 9)(en 9)
 5  en-gb-x-rp      --/M      English_(Received_Pronunciation) gmw/en-GB-x-rp       (en-gb 4)(en 5)
 2  en-us           --/M      English_(America)  gmw/en-US            (en 3)
 5  fr-be           --/M      French_(Belgium)   roa/fr-BE            (fr 8)
 5  fr-ch           --/M      French_(Switzerland) roa/fr-CH            (

## Step 11: Verify Training Data

In [25]:
import pandas as pd

# Read and verify metadata
if METADATA_CSV.exists():
    df = pd.read_csv(str(METADATA_CSV), sep="|", header=None, names=["audio", "text"])

    print(f"📊 Training Data Summary:")
    print(f"   Total samples: {len(df)}")
    print(f"\n📝 First 5 entries:")
    print(df.head())

    # Check if audio files exist
    print(f"\n🔍 Checking audio files...")
    missing = [a for a in df["audio"].head(5) if not (AUDIO_DIR / str(a)).exists()]

    if missing:
        print(f"   ⚠️ Missing files: {missing}")
    else:
        print(f"   ✅ All checked files exist!")
else:
    print(f"❌ Metadata CSV not found at: {METADATA_CSV}")

📊 Training Data Summary:
   Total samples: 30

📝 First 5 entries:
                               audio  \
0  voice_🇵🇱 Polski_1772658337076.wav   
1  voice_🇵🇱 Polski_1772658379643.wav   
2  voice_🇵🇱 Polski_1772658437218.wav   
3  voice_🇵🇱 Polski_1772726876909.wav   
4  voice_🇵🇱 Polski_1772726940490.wav   

                                                text  
0  Słońce już wysoko, a poranne mgły powoli znika...  
1  Muszę dziś załatwić kilka ważnych spraw urzędo...  
2  Zastanawiam się, czy wziąć dziś parasol, bo po...  
3  Dziś rześki wiatr wieje przez rozległe pola, n...  
4  Poranny, rześki deszcz zrosił szybko wszystkie...  

🔍 Checking audio files...
   ✅ All checked files exist!


## Step 12: Start Training! 🚀

**This will take 2-4 hours with a checkpoint, or 8-12+ hours without.**

You can close this tab and come back later - the training will continue in the background.

In [26]:
# Start Piper training
print("🚀 Starting Piper training...\n")
print("   ⏱️ This will take several hours.")
print("   💡 You can close this tab - training continues in background.")
print("   📊 Monitor progress below...\n")

training_command = f"""
!python3 -m piper.train fit \
  --data.voice_name "{VOICE_NAME}" \
  --data.csv_path "{str(METADATA_CSV)}" \
  --data.audio_dir "{str(AUDIO_DIR)}" \
  --model.sample_rate {SAMPLE_RATE_HZ} \
  --data.espeak_voice "{ESPEAK_VOICE}" \
  --data.cache_dir "{str(CACHE_DIR)}" \
  --data.config_path "{str(CONFIG_PATH)}" \
  --data.batch_size {BATCH_SIZE}
"""

# Conditionally add --ckpt_path if CKPT_URL is not empty
if CKPT_URL:
    training_command += f" \
  --ckpt_path "{CKPT_URL}""

# Execute the constructed command
get_ipython().system(training_command)

print("\n\n✅ Training complete!")

🚀 Starting Piper training...

   ⏱️ This will take several hours.
   💡 You can close this tab - training continues in background.
   📊 Monitor progress below...

/usr/local/lib/python3.12/dist-packages/lightning/fabric/utilities/seed.py:44: No seed found, seed set to 0
Seed set to 0
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/piper1-gpl/src/piper/tra

## Step 13: Export to ONNX Format

In [ ]:
# Find the latest checkpoint
import glob

checkpoint_pattern = "/content/piper1-gpl/lightning_logs/version_*/checkpoints/*.ckpt"
checkpoints = sorted(glob.glob(checkpoint_pattern))

if checkpoints:
    latest_checkpoint = checkpoints[-1]
    print(f"📦 Found checkpoint: {latest_checkpoint}")

    output_onnx = "/content/drive/MyDrive/piper_training/model.onnx"

    print(f"\n🔄 Exporting to ONNX format...")
    !python3 -m piper.train.export_onnx \
      --checkpoint "{latest_checkpoint}" \
      --output-file "{output_onnx}"

    print(f"\n✅ Model exported to: {output_onnx}")
else:
    print("❌ No checkpoint found. Make sure training completed successfully.")

## Step 14: Copy Config File

In [ ]:
# Copy the config JSON file
import shutil

if CONFIG_PATH.exists():
    config_dest = "/content/drive/MyDrive/piper_training/model.onnx.json"
    shutil.copy2(str(CONFIG_PATH), config_dest)

    print(f"✅ Config copied to: {config_dest}")
    print(f"\n📦 Your trained model files:")
    print(f"   1. model.onnx")
    print(f"   2. model.onnx.json")
    print(f"\n   Both files are in: /content/drive/MyDrive/piper_training/")
else:
    print(f"❌ Config file not found at: {CONFIG_PATH}")

## Step 15: Test Your Voice Model!

In [ ]:
from IPython.display import Audio, display
import subprocess

# Test text based on detected language
test_texts = {
    "de": "Hallo! Dies ist meine trainierte Stimme mit Piper Text to Speech.",
    "en-us": "Hello! This is my trained voice using Piper text to speech.",
    "pl": "Cześć! To jest mój wyszkolony głos używający Piper text to speech."
}

test_text = test_texts.get(ESPEAK_VOICE, "Hello, this is a test.")

print(f"🎤 Testing your voice model...")
print(f"   Language: {ESPEAK_VOICE}")
print(f"   Text: {test_text}")

output_audio = "/content/test_output.wav"
model_path = "/content/drive/MyDrive/piper_training/model.onnx"

# Generate speech
!echo "{test_text}" | /content/piper1-gpl/piper \
  --model "{model_path}" \
  --output_file "{output_audio}"

print(f"\n🔊 Generated audio:")
display(Audio(output_audio))

print(f"\n💡 If the quality is not good enough:")
print(f"   - Record more training data (60+ minutes recommended)")
print(f"   - Ensure audio quality is consistent")
print(f"   - Try training for more epochs")

## ✅ Training Complete!

### Your trained model files:
- `model.onnx` - The trained neural network
- `model.onnx.json` - Configuration file

### Location:
`/content/drive/MyDrive/piper_training/`

### Next Steps:
1. Download both files from Google Drive
2. Use them with Piper TTS in your Android app
3. Or use with desktop Piper: https://github.com/rhasspy/piper

### Integration with Android:
```kotlin
// Example usage in your Speech2Text app
val piperTts = PiperTTS(
    modelPath = "path/to/model.onnx",
    configPath = "path/to/model.onnx.json"
)

val audioData = piperTts.synthesize("Your text here")
```

### Congratulations! 🎉
You've successfully trained a custom voice model with Piper TTS!